# Блокнот 20. Итоговый проект: детектор животных саванны от данных до ноутбука в полевом лагере

Курс «Машинное обучение с нуля», модуль 20. Разделы: 1 — урок 20.1 (данные), 2 — урок 20.2 (обучение и разбор ошибок), 3 — урок 20.3 (экспорт, ускорение и замер). Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Задача заповедника: находить на снимках буйволов, слонов, носорогов и зебр на ноутбуке без видеокарты, не дольше 100 мс на снимок. В Colab включите видеокарту для обучения: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без неё всё работает, но обучение короче: 5 эпох на снимках 320 × 320 (≈5 минут) вместо 30 эпох на 640 × 640.

Блокнот ставит onnx, onnxruntime, openvino и nncf (≈150 МБ) и скачивает набор African Wildlife (≈100 МБ) и веса YOLO26n (≈5 МБ).

In [ ]:
%pip install -q ultralytics onnx onnxruntime onnxscript openvino nncf

In [ ]:
import logging
import os
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from matplotlib.patches import Rectangle
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

logging.getLogger("nncf").setLevel(logging.ERROR)
device = "cuda" if torch.cuda.is_available() else "cpu"
DEV = 0 if device == "cuda" else "cpu"
EPOCHS, IMGSZ = (30, 640) if device == "cuda" else (5, 320)
print("устройство:", device, "| ultralytics", ultralytics.__version__, f"| обучение: {EPOCHS} эпох, {IMGSZ} × {IMGSZ}")


def draw(ax, boxes, labels, color):
    for (x1, y1, x2, y2), label in zip(boxes, labels):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=color, lw=2))
        ax.text(x1, y1 - 4, label, color="white", fontsize=8, bbox=dict(fc=color, ec="none", pad=1))

## 1. Данные: скачиваем и проверяем (урок 20.1)

`check_det_dataset` читает описание набора и при первом запуске скачивает его. Снимки лежат в `images/train`, `images/val`, `images/test`, разметка YOLO — в таких же папках `labels/...`: на снимок один файл `.txt`, строка на объект — `класс x_центра y_центра ширина высота` в долях снимка (урок 16.1).

In [ ]:
data = check_det_dataset("african-wildlife.yaml")
names = data["names"]
root = Path(data["path"])


def images(split):  # снимки части набора; у трёх файлов расширение .JPG заглавными
    return sorted(p for p in (root / "images" / split).iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))


print("набор:", root)
print("классы:", names)
print({split: len(images(split)) for split in ("train", "val", "test")})

**Задание 1.** Допишите `read_labels(path)`: прочитайте файл разметки YOLO и верните список кортежей `(класс, ширина, высота)` — класс целым числом, ширина и высота — дробными. Пустые строки пропускайте; если файла нет, верните пустой список.

In [ ]:
def read_labels(path):
    if not os.path.exists(path):
        return []
    ...  # ваш код


rows = []
for split in ("train", "val", "test"):
    for img in images(split):
        objs = read_labels(root / "labels" / split / (img.stem + ".txt"))
        if not objs:
            rows.append({"split": split, "image": img.name, "class": None, "area": None})
        for c, w, h in objs:
            rows.append({"split": split, "image": img.name, "class": names[c], "area": w * h})
labels = pd.DataFrame(rows)
assert len(labels.dropna()) > 2000 and set(labels["class"].dropna()) == set(names.values())

table = labels.dropna().pivot_table(index="class", columns="split", values="image", aggfunc="count")[["train", "val", "test"]]
print("объектов по классам:\n", table, "\n")
print("снимков без разметки:", labels["class"].isna().sum())
print(f"мелких рамок (меньше 1 % снимка): {(labels['area'] < 0.01).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
table.plot.bar(ax=axes[0], rot=0)
axes[0].set_ylim(0, table.values.max() * 1.25)
axes[0].legend(ncol=3, loc="upper left")
axes[0].set_title("объектов каждого класса")
axes[1].hist(np.sqrt(labels["area"].dropna()), bins=40, color="tab:blue")
axes[1].set_xlabel("√(площадь рамки / площадь снимка)")
axes[1].set_title("размеры рамок: от мелких к крупным")
plt.show()

Посмотрим на снимки глазами — аудит не заканчивается цифрами.

In [ ]:
rng = np.random.default_rng(0)
sample = rng.choice(images("train"), 8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, img in zip(axes.flat, sample):
    im = cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    boxes, labs = [], []
    with open(root / "labels" / "train" / (img.stem + ".txt")) as f:
        for line in f:
            if line.split():
                c, x, y, w, h = map(float, line.split()[:5])
                boxes.append(((x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H))
                labs.append(names[int(c)])
    ax.imshow(im)
    draw(ax, boxes, labs, "tab:green")
    ax.set_title(img.name, fontsize=8)
    ax.axis("off")
plt.show()

**Базовое решение.** Что умеет готовая YOLO26n, обученная на COCO? Слоны и зебры там есть, а носорогов и буйволов нет. Посмотрим, кем модель COCO считает носорогов на тестовых снимках.

In [ ]:
coco = YOLO("yolo26n.pt")
rhino_imgs = sorted(labels[(labels["split"] == "test") & (labels["class"] == "rhino")]["image"].unique())
seen = Counter()
for img in rhino_imgs:
    r = coco.predict(str(root / "images" / "test" / img), conf=0.25, verbose=False)[0]
    seen.update(r.names[int(c)] for c in r.boxes.cls)
print(f"снимков с носорогами: {len(rhino_imgs)}; что на них нашла модель COCO:", seen.most_common(6))

Носорогов модель COCO называет чужими именами — это и есть причина дообучать. А вот слонов и зебр она уже знает: при переносе обучения эти знания пригодятся (урок 13.7).

## 2. Обучение и разбор ошибок (урок 20.2)

Дообучаем предобученную YOLO26n. На процессоре — 5 эпох на 320 × 320 (около 5 минут), на видеокарте — 30 эпох на 640 × 640. Результаты лягут в папку `runs_m20/wildlife`.

In [ ]:
model = YOLO("yolo26n.pt")
t0 = time.time()
model.train(data="african-wildlife.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=16, workers=2 if device == "cuda" else 0,
            device=DEV, project=os.path.abspath("runs_m20"), name="wildlife", exist_ok=True, verbose=False)
save_dir = Path(model.trainer.save_dir)
print(f"обучение: {time.time() - t0:.0f} с")

df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for col in ("train/box_loss", "train/cls_loss", "val/box_loss", "val/cls_loss"):
    axes[0].plot(df["epoch"], df[col], "-o", ms=3, label=col)
axes[0].set_title("потери")
axes[0].legend(fontsize=8)
for col in ("metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    axes[1].plot(df["epoch"], df[col], "-o", ms=3, label=col)
axes[1].set_title("метрики на валидации")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("эпоха")
plt.show()

Лучшие веса — `best.pt`. Метрики по классам на валидации: AP50-95 для каждого класса и среднее mAP.

In [ ]:
best = YOLO(save_dir / "weights" / "best.pt")
common = dict(data="african-wildlife.yaml", imgsz=IMGSZ, batch=16, workers=0, device=DEV, verbose=False,
              project=os.path.abspath("runs_m20"), exist_ok=True)
val = best.val(split="val", name="val", **common)
per_class = pd.DataFrame({"AP50-95 (val)": val.box.maps[:len(names)]}, index=list(names.values())).round(3)
print(per_class, f"\nmAP50 {val.box.map50:.3f}, mAP50-95 {val.box.map:.3f}")

**Разбор ошибок.** mAP говорит «сколько», а не «почему». Разложим каждое предсказание с уверенностью ≥ 0,25 по типам ошибок (урок 20.2): верно, дубль, ошибка класса, ошибка места, фон — и отдельно посчитаем пропуски.

**Задание 2.** Допишите `error_types(preds, gts)`. `preds` — список `(класс, уверенность, рамка)`, `gts` — список `(класс, рамка)`, рамки `(x1, y1, x2, y2)`; функция `iou(a, b)` уже есть. Предсказания перебирайте **по убыванию уверенности**, и для каждого:
1. есть объект того же класса с IoU ≥ 0,5, ещё не занятый, — `"верно"` (возьмите такой с наибольшим IoU и пометьте занятым);
2. иначе есть объект того же класса с IoU ≥ 0,5 (уже занятый) — `"дубль"`;
3. иначе есть объект **другого** класса с IoU ≥ 0,5 — `"класс"`;
4. иначе есть объект того же класса с IoU от 0,1 до 0,5 — `"место"`;
5. иначе — `"фон"`.

В конце добавьте `"пропуск"` — число незанятых объектов. Верните `Counter`.

In [ ]:
def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def error_types(preds, gts):
    out = Counter()
    used = [False] * len(gts)
    ...  # ваш код
    out["пропуск"] = used.count(False)
    return out


gt_one = [(0, (0, 0, 10, 10)), (1, (20, 0, 30, 10))]
check = error_types([(0, 0.9, (0, 0, 10, 10)), (0, 0.8, (1, 0, 10, 10)), (0, 0.7, (20, 0, 30, 10)),
                     (1, 0.6, (50, 50, 60, 60))], gt_one)
assert check == Counter({"верно": 1, "дубль": 1, "класс": 1, "фон": 1, "пропуск": 1}), check

In [ ]:
errors, per_image = Counter(), []
for img in images("val"):
    r = best.predict(str(img), imgsz=IMGSZ, conf=0.25, device=DEV, verbose=False)[0]
    preds = [(int(c), float(s), tuple(b)) for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy.tolist())]
    H, W = r.orig_shape
    gts = []
    for c, x, y, w, h in [(int(l.split()[0]), *map(float, l.split()[1:5]))
                          for l in open(root / "labels" / "val" / (img.stem + ".txt")) if l.split()]:
        gts.append((c, ((x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H)))
    e = error_types(preds, gts)
    errors += e
    per_image.append((sum(v for k, v in e.items() if k != "верно"), img, r, gts))
print("типы предсказаний на валидации:", dict(errors))

order = ["верно", "дубль", "класс", "место", "фон", "пропуск"]
plt.figure(figsize=(7, 3))
plt.bar(order, [errors[k] for k in order], color=["tab:green"] + ["tab:orange"] * 5)
plt.title("предсказания и пропуски на валидации (уверенность ≥ 0,25)")
plt.show()

Четыре снимка валидации с наибольшим числом ошибок: зелёные — разметка, красные — предсказания модели.

In [ ]:
worst = sorted(per_image, key=lambda t: -t[0])[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (n_err, img, r, gts) in zip(axes, worst):
    ax.imshow(cv2.cvtColor(cv2.imread(str(img)), cv2.COLOR_BGR2RGB))
    draw(ax, [b for _, b in gts], [names[c] for c, _ in gts], "tab:green")
    draw(ax, r.boxes.xyxy.tolist(), [f"{names[int(c)]} {s:.2f}" for c, s in zip(r.boxes.cls, r.boxes.conf)], "tab:red")
    ax.set_title(f"{img.name}: ошибок {n_err}", fontsize=9)
    ax.axis("off")
plt.show()

Матрица ошибок при рабочем пороге 0,25 показывает, какие классы модель путает между собой (строки — предсказание, столбцы — истина, урок 17.8).

In [ ]:
check25 = best.val(split="val", name="val_conf25", conf=0.25, plots=True, **common)
plt.figure(figsize=(7, 6))
plt.imshow(plt.imread(Path(check25.save_dir) / "confusion_matrix.png"))
plt.axis("off")
plt.show()

**Экзамен.** Тестовую выборку смотрим один раз — когда модель и настройки уже выбраны по валидации.

In [ ]:
test = best.val(split="test", name="test", **common)
per_class["AP50-95 (test)"] = np.round(test.box.maps[:len(names)], 3)
print(per_class, f"\nтест: mAP50 {test.box.map50:.3f}, mAP50-95 {test.box.map:.3f}")
assert test.box.map50 > 0.5

## 3. Экспорт, ускорение и замер (урок 20.3)

Целевое железо — ноутбук без видеокарты. Сравним четыре варианта модели: PyTorch, ONNX Runtime, OpenVINO FP32 и OpenVINO INT8 (калибровка по 300 снимкам **train**). Для каждого — mAP на тесте и время сети на снимок.

Экспортируем с настройками по умолчанию: в файл попадает сырой выход сети, а NMS выполняет Ultralytics — так же, как при валидации PyTorch-модели. Голову без NMS (`nms=False`, урок 19.2) после короткого дообучения лучше не брать: у автора курса она дала на тесте mAP50-95 0,714 против 0,773 — ей нужно больше эпох.

In [ ]:
paths = {
    "PyTorch": save_dir / "weights" / "best.pt",
    "ONNX Runtime": best.export(format="onnx", imgsz=IMGSZ, verbose=False),
    "OpenVINO FP32": best.export(format="openvino", imgsz=IMGSZ, verbose=False),
    "OpenVINO INT8": best.export(format="openvino", imgsz=IMGSZ, quantize=8, data="african-wildlife.yaml",
                                 split="train", fraction=300 / 1052, verbose=False),
}
test_imgs = images("test")[:30]
results = {}
for name, path in paths.items():
    m = YOLO(path, task="detect")
    dev = DEV if name == "PyTorch" else "cpu"
    mp = m.val(split="test", name=f"test_{name.replace(' ', '_')}", **{**common, "batch": 1, "device": dev}).box.map
    for img in test_imgs[:3]:
        m.predict(str(img), imgsz=IMGSZ, device=dev, verbose=False)  # прогрев
    ms = np.median([m.predict(str(img), imgsz=IMGSZ, device=dev, verbose=False)[0].speed["inference"] for img in test_imgs])
    results[name] = (mp, ms)
    print(f"{name:14s} mAP50-95 {mp:.3f}, сеть {ms:5.1f} мс на снимок")

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, k, color, fmt, title in ((axes[0], 0, "tab:green", "{:.3f}", "mAP50-95 на тесте (больше — лучше)"),
                                 (axes[1], 1, "tab:blue", "{:.1f}", "время сети на снимок, мс (меньше — лучше)")):
    vals = [v[k] for v in results.values()][::-1]
    ax.barh(list(results)[::-1], vals, color=color)
    for y, v in enumerate(vals):
        ax.text(v, y, " " + fmt.format(v), va="center", fontsize=9)
    ax.set_xlim(0, max(vals) * 1.25)
    ax.set_title(title)
plt.tight_layout()
plt.show()

**Задание 3.** Допишите `pick(results, max_drop, budget_ms)`: `results` — словарь «вариант → (mAP, мс)», где есть `"PyTorch"`. Годится вариант, у которого mAP не ниже, чем у PyTorch, больше чем на `max_drop`, и время не больше `budget_ms`. Верните имя **самого быстрого** подходящего варианта или `None`, если таких нет.

In [ ]:
def pick(results, max_drop, budget_ms):
    ...  # ваш код


assert pick({"PyTorch": (0.70, 40), "A": (0.69, 20), "B": (0.60, 5)}, 0.02, 100) == "A"
assert pick({"PyTorch": (0.70, 40)}, 0.01, 30) is None
choice = pick(results, max_drop=0.01, budget_ms=100)
print("выбор для полевого ноутбука:", choice)

**Паспорт модели.** Последний шаг проекта — коротко записать, что получилось, на каких данных и с какими ограничениями. Такой паспорт (model card) читают те, кто будет пользоваться моделью.

In [ ]:
mp, ms = results[choice] if choice else results["PyTorch"]
card = f"""Детектор животных саванны — паспорт модели
Модель: YOLO26n, дообучена {EPOCHS} эпох на {IMGSZ} × {IMGSZ}; формат для работы: {choice or 'PyTorch'}
Данные: African Wildlife (архив Ultralytics, источник — Kaggle, где лицензия на снимки не указана: только для учёбы)
Классы: {', '.join(names.values())}
Тест (227 снимков): mAP50-95 {mp:.3f}; AP50-95 по классам у исходной модели: {dict(zip(names.values(), per_class['AP50-95 (test)']))}
Скорость: {ms:.1f} мс на снимок (сеть) на {'видеокарте' if device == 'cuda' and choice == 'PyTorch' else 'процессоре'} этого компьютера
Ограничения: снимки днём, крупные животные; ночные снимки фотоловушек, другие виды и мелкие объекты не проверялись"""
print(card)

Что попробовать дальше: больше эпох и разрешение 640 (в Colab с видеокартой), модель YOLO26s, свои снимки в тесте — и посмотреть, как меняются ошибки по типам. Это и есть настоящая работа над проектом: данные → модель → ошибки → данные.